In [3]:
import csv
from pathlib import Path


def load_caiso_timelines(time_data_dir="."):
    time_data_dir = Path(time_data_dir)
    demand_file = time_data_dir / "20260406_20260407_SLD_FCST_DAM_20260408_13_08_29_v1.csv"
    renewable_file = time_data_dir / "20260406_20260407_SLD_REN_FCST_DAM_20260408_13_08_02_v1.csv"

    def build_hourly_list(csv_path, filters):
        hourly_values = [None] * 24

        with csv_path.open(newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if any(row[key] != value for key, value in filters.items()):
                    continue

                hour_index = int(row["OPR_HR"]) - 1
                hourly_values[hour_index] = float(row["MW"])

        missing_hours = [hour for hour, value in enumerate(hourly_values) if value is None]
        if missing_hours:
            raise ValueError(f"Missing data for hours: {missing_hours} with filters {filters}")

        return hourly_values

    def scale_to_unit_interval(values):
        max_value = max(values)
        if max_value == 0:
            return [0.0 for _ in values]
        return [value / max_value for value in values]

    demand_timeline = build_hourly_list(
        demand_file,
        {"TAC_AREA_NAME": "PGE-TAC"},
    )
    solar_timeline = build_hourly_list(
        renewable_file,
        {"TRADING_HUB": "NP15", "RENEWABLE_TYPE": "Solar"},
    )
    wind_timeline = build_hourly_list(
        renewable_file,
        {"TRADING_HUB": "NP15", "RENEWABLE_TYPE": "Wind"},
    )

    demand_timeline = scale_to_unit_interval(demand_timeline)
    solar_timeline = scale_to_unit_interval(solar_timeline)
    wind_timeline = scale_to_unit_interval(wind_timeline)

    return demand_timeline, solar_timeline, wind_timeline


demand_timeline, solar_timeline, wind_timeline = load_caiso_timelines(".")
demand_timeline, solar_timeline, wind_timeline


([0.8186355237458459,
  0.7892294745579047,
  0.7628280925262919,
  0.7516888386784405,
  0.7635592913300242,
  0.8049304867083208,
  0.870919531901018,
  0.9146185414889447,
  0.8717750674382679,
  0.7827985481421951,
  0.700026514190856,
  0.6295753447668233,
  0.5873057959033104,
  0.5739078953002361,
  0.5954346186402996,
  0.6429386616426941,
  0.7263669629888244,
  0.8450681958163571,
  0.9539303582544769,
  1.0,
  0.9981127166011772,
  0.963940700435755,
  0.9048469917097866,
  0.8479213533106508],
 [0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.004400946112859818,
  0.2486951104220924,
  0.704575172868439,
  0.8814424961152143,
  0.9508144466942193,
  0.9930309297768377,
  1.0,
  0.9998768459523972,
  0.9923028720248337,
  0.9830047414308327,
  0.9612608076732216,
  0.8392477460998199,
  0.2981052387558544,
  0.010511560180674232,
  0.0,
  0.0,
  0.0,
  0.0],
 [0.31418721193942495,
  0.3921190284585559,
  0.4896389640793035,
  0.5883202867803058,
  0.6504316336235277,
  0.6639841